In [1]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
!ls -lh /content/drive/MyDrive/bosch_yolo_jpg.zip

Tue Nov 10 06:49:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import shutil
shutil.copy("/content/drive/MyDrive/bosch_yolo_jpg.zip", "/content/")
!unzip -q /content/bosch_yolo_jpg.zip -d /content/
!pip install -q ultralytics
!ls /content/bosch_yolo_jpg/images/train | wc -l
!ls /content/bosch_yolo_jpg/images/val | wc -l
!cat /content/bosch_yolo_jpg/data.yaml

4228
865
path: /content/bosch_yolo_jpg
train: images/train
val: images/val
names:
  0: red
  1: yellow
  2: green
  3: 'off'


In [ ]:
!yolo detect train data=/content/bosch_yolo_jpg/data.yaml model=yolo11n.pt imgsz=1280 epochs=30 batch=8 workers=2 seed=0 patience=10 project=/content/drive/MyDrive/tl_runs name=y11n_1280_e30 exist_ok=True

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bosch_yolo_jpg/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0,

In [4]:
import glob, shutil
BEST = "/content/drive/MyDrive/tl_runs/y11n_1280_e30/weights/best.pt"
print(glob.glob(BEST))
shutil.copy(BEST, "/content/drive/MyDrive/tl_runs/traffic_lights_y11n_1280_best.pt")   # Kişi 3'e gönderilecek kopya
!yolo detect val model={BEST} data=/content/bosch_yolo_jpg/data.yaml imgsz=1280 batch=8 plots=True project=/content/drive/MyDrive/tl_runs name=val_best exist_ok=True

['/content/drive/MyDrive/tl_runs/y11n_1280_e30/weights/best.pt']
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2224.9±406.2 MB/s, size: 199.4 KB)
val: Scanning /content/bosch_yolo_jpg/labels/val.cache... 865 images, 382 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 865/865 106.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 6.5it/s 16.8s
                   all        865       1756      0.679      0.503      0.533      0.293
                   red        337        919       0.88        0.7      0.785      0.448
                yellow         29         56      0.703      0.446      0.447       0.22
                 green        267        711      0.792      0.723      0.731      0.394
                   off         57         70      0.344  

In [5]:
import numpy as np, cv2, csv, shutil
from pathlib import Path
from ultralytics import YOLO

DST = Path("/content/bosch_yolo_jpg"); W, H = 1280, 720
C = ["red", "yellow", "green", "off"]
OUT = Path("/content/drive/MyDrive/tl_runs/finetuned_eval")
(OUT/"pred_labels").mkdir(parents=True, exist_ok=True); (OUT/"samples").mkdir(exist_ok=True)

def gt_boxes(stem):
    out = []
    for ln in (DST/"labels/val"/f"{stem}.txt").read_text().splitlines():
        c, cx, cy, bw, bh = map(float, ln.split())
        out.append([int(c), (cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H])
    return np.array(out).reshape(-1, 5)

def iou(a, b):
    x1 = np.maximum(a[0], b[:,0]); y1 = np.maximum(a[1], b[:,1])
    x2 = np.minimum(a[2], b[:,2]); y2 = np.minimum(a[3], b[:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[:,2]-b[:,0])*(b[:,3]-b[:,1]) - inter + 1e-9)

def size_bin(w): return "<10px" if w < 10 else "10-20px" if w < 20 else ">=20px"

model = YOLO(BEST)
R = {"ag": {"TP":0,"FP":0}, "cw": {"TP":0,"FP":0}}
size = {"ag": {}, "cw": {}}; state = {}
cm = np.zeros((4, 5), int)   # rows: GT state, cols: predicted state + "missed"
speeds, rows = [], []
for i, r in enumerate(model.predict(source=str(DST/"images/val"), batch=1, imgsz=1280, conf=0.25,
                                    stream=True, verbose=False)):
    stem = Path(r.path).stem; g = gt_boxes(stem); speeds.append(r.speed["inference"])
    cf = r.boxes.conf.cpu().numpy(); o = np.argsort(-cf)
    xyxy = r.boxes.xyxy.cpu().numpy()[o]; cl = r.boxes.cls.cpu().numpy().astype(int)[o]; cf = cf[o]
    lines = []
    for b, k, c in zip(xyxy, cl, cf):
        lines.append(f"{k} {(b[0]+b[2])/2/W:.6f} {(b[1]+b[3])/2/H:.6f} {(b[2]-b[0])/W:.6f} {(b[3]-b[1])/H:.6f} {c:.4f}")
        rows.append([stem, i, *[round(float(v), 1) for v in b], C[k], round(float(c), 4)])
    (OUT/"pred_labels"/f"{stem}.txt").write_text("\n".join(lines))
    for mode in ["ag", "cw"]:            # ag = class-agnostic (baseline ile karşılaştırma), cw = class-aware
        used = np.zeros(len(g), bool); mp = {}
        for b, k in zip(xyxy, cl):
            if len(g):
                ious = iou(b, g[:,1:]); ious[used] = 0
                if mode == "cw": ious[g[:,0] != k] = 0
                m = ious.argmax()
                if ious[m] >= 0.5: used[m] = True; R[mode]["TP"] += 1; mp[m] = k; continue
            R[mode]["FP"] += 1
        for m, gb in enumerate(g):
            d = size[mode]; key = size_bin(gb[3]-gb[1]); d.setdefault(key, [0,0]); d[key][0] += int(used[m]); d[key][1] += 1
            if mode == "cw":
                s = C[int(gb[0])]; state.setdefault(s, [0,0]); state[s][0] += int(used[m]); state[s][1] += 1
            else:
                cm[int(gb[0]), mp.get(m, 4)] += 1
    if i % 120 == 0 and len(g):          # baseline ile aynı örnek görüntüler
        im = r.plot(line_width=1, font_size=8)
        for gb in g: cv2.rectangle(im, (int(gb[1])-3, int(gb[2])-3), (int(gb[3])+3, int(gb[4])+3), (0,255,0), 1)
        cv2.imwrite(str(OUT/"samples"/f"{stem}.jpg"), im)

with open(OUT/"val_predictions.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["image", "frame_index", "x1", "y1", "x2", "y2", "class_name", "confidence"]); w.writerows(rows)
shutil.make_archive(str(OUT/"pred_labels"), "zip", OUT/"pred_labels")

n_gt = sum(v[1] for v in size["ag"].values())
L = [f"Fine-tuned YOLO11n | imgsz=1280 conf=0.25 IoU>=0.5 | val: 865 images, {n_gt} lights",
     f"Mean inference: {np.mean(speeds):.1f} ms/image (Tesla T4, model only)"]
for mode, name in [("ag", "CLASS-AGNOSTIC (comparable to baseline)"), ("cw", "CLASS-AWARE (correct state required)")]:
    tp, fp = R[mode]["TP"], R[mode]["FP"]
    L += [f"\n--- {name} ---", f"Precision {tp/max(tp+fp,1):.3f} | Recall {tp/n_gt:.3f} | TP {tp} FP {fp}", "Recall by size:"]
    for k in sorted(size[mode]): L.append(f"   {k:8s} {size[mode][k][0]/size[mode][k][1]:.3f}  ({size[mode][k][0]}/{size[mode][k][1]})")
L.append("\nRecall by state (class-aware):")
for k in C: L.append(f"   {k:7s} {state[k][0]/state[k][1]:.3f}  ({state[k][0]}/{state[k][1]})")
L.append("\nState confusion (rows = GT, cols = predicted; only class-agnostic matches, 'missed' = not detected):")
L.append("          " + "".join(f"{x:>8s}" for x in C + ["missed"]))
for gi, gname in enumerate(C): L.append(f"   {gname:7s}" + "".join(f"{v:8d}" for v in cm[gi]))
txt = "\n".join(L); print(txt); (OUT/"finetuned_results.txt").write_text(txt)

Fine-tuned YOLO11n | imgsz=1280 conf=0.25 IoU>=0.5 | val: 865 images, 1756 lights
Mean inference: 15.9 ms/image (Tesla T4, model only)

--- CLASS-AGNOSTIC (comparable to baseline) ---
Precision 0.888 | Recall 0.684 | TP 1201 FP 151
Recall by size:
   10-20px  0.894  (420/470)
   <10px    0.523  (530/1013)
   >=20px   0.919  (251/273)

--- CLASS-AWARE (correct state required) ---
Precision 0.868 | Recall 0.668 | TP 1173 FP 179
Recall by size:
   10-20px  0.879  (413/470)
   <10px    0.513  (520/1013)
   >=20px   0.879  (240/273)

Recall by state (class-aware):
   red     0.687  (631/919)
   yellow  0.411  (23/56)
   green   0.717  (510/711)
   off     0.129  (9/70)

State confusion (rows = GT, cols = predicted; only class-agnostic matches, 'missed' = not detected):
               red  yellow   green     off  missed
   red         628       6       2       8     275
   yellow        3      23       0       0      30
   green         1       0     510       0     200
   off           0   

1029

In [6]:
import shutil
shutil.make_archive("/content/drive/MyDrive/tl_runs/finetuned_eval/val_gt_labels", "zip", "/content/bosch_yolo_jpg/labels/val")
!ls -lh /content/drive/MyDrive/tl_runs/finetuned_eval/

total 585K
-rw------- 1 root root 1.1K Nov 10 10:16 finetuned_results.txt
drwx------ 2 root root 4.0K Nov 10 10:16 pred_labels
-rw------- 1 root root 215K Nov 10 10:16 pred_labels.zip
drwx------ 2 root root 4.0K Nov 10 10:16 samples
-rw------- 1 root root 219K Nov 10 10:18 val_gt_labels.zip
-rw------- 1 root root 142K Nov 10 10:16 val_predictions.csv
